In [1]:
# Imports

In [2]:
import pandas as pd 
import json
from typing import List
import os
from os import listdir
import matplotlib.pyplot as plt
import spotipy
import spotipy.util as util
from spotipy.oauth2 import SpotifyClientCredentials
import plotly.express as px
import plotly.io as pio
from datetime import datetime
import pytz 
from collections import Counter

from utilis import (
    get_streamings,
    minsec_to_seconds,
    format_time,
    generate_top_10_songs_by_year,
    save_cache,
    search_track_id,
    get_audio_features
)

In [3]:
# Load cleaned streaming data
STREAMING_DATA_PATH = 'wrapped_data/streaming_data.csv'
streaming_data = pd.read_csv(STREAMING_DATA_PATH)

# Convert 'ts to datetime format
streaming_data['ts'] = pd.to_datetime(streaming_data['ts'])

In [4]:
# Title: Display Top 10 Songs per Year

# Description:
# Generates a dictionary of top 10 songs for each year using streaming history, 
# iterates through each year to print a clean, ranked table, 
# assigns song rank, renames columns for clarity, 
# selects relevant metadata including listening duration and Spotify URI, 
# and displays the final table for each year's Wrapped summary.


In [5]:
# Top 10 songs by year (resembling Spotify Wrapped)
top_10_songs_by_year = generate_top_10_songs_by_year(streaming_data)

for year, df in top_10_songs_by_year.items():
    print(f"\nWrapped {year} – Top 10 Songs:")

    if df.empty:
        print("No data for this year.")
        continue

    # Convert 'listening_length' (MM:SS) to total seconds
    df['listening_seconds'] = df['listening_length'].apply(
        lambda x: int(x.split(":")[0]) * 60 + int(x.split(":")[1])
    )

    # Clean DataFrame and sort by listening time (most to least)
    top_10_songs = (
        df.copy()
        .sort_values(by='listening_seconds', ascending=False)
        .head(10)
        .assign(rank=range(1, 11))
        .rename(columns={
            'master_metadata_track_name': 'track_name',
            'master_metadata_album_artist_name': 'artist_name',
            'master_metadata_album_album_name': 'album_name',
            'spotify_track_uri': 'spotify_track_uri',
            'play_count': 'total_play_count'  
        })
        [['rank', 'track_name', 'artist_name', 'album_name', 'total_play_count', 'listening_length', 'spotify_track_uri']]
    )

    display(top_10_songs)


Processing Wrapped 2017...
Processing Wrapped 2018...
Processing Wrapped 2019...
Processing Wrapped 2020...
Processing Wrapped 2021...
Processing Wrapped 2022...
Processing Wrapped 2023...
Processing Wrapped 2024...

Wrapped 2017 – Top 10 Songs:


,rank,track_name,artist_name,album_name,total_play_count,listening_length,spotify_track_uri
2861,1,Living In A Memory,The Growlers,Hung At Heart,130,347:59,spotify:track:2b9D3N4pnBUbYXVi7vbWy4
4351,2,Someday,The Growlers,Hung At Heart,139,319:17,spotify:track:5uplevkaUGRLExATEbSQiR
4465,3,Still Beating,Mac DeMarco,This Old Dog,109,273:18,spotify:track:2N4idqj9TT3HnH2OFT9j0v
1423,4,Everlasting Light,The Black Keys,Brothers,98,266:02,spotify:track:6dU5RxthbuaN31bRbEDlNw
5068,5,Trouble,Cage The Elephant,Tell Me I'm Pretty,103,242:58,spotify:track:5n0CTysih20NYdT2S0Wpe8
4624,6,Take It or Leave It,Cage The Elephant,Melophobia,92,241:07,spotify:track:43O3Iu8mDJy10i6k8SVRXX
4952,7,Threat of Joy,The Strokes,Future Present Past,93,228:01,spotify:track:6YQeOwMMAkB9MV9yMWmrjh
5265,8,Wandering eyes,The Growlers,Are You In Or Out?,110,170:27,spotify:track:0ZcZdY4cVfcjZYTUuKgj0E
2568,9,Joy Ride,The Killers,Day & Age,104,165:38,spotify:track:07SgR5jvYFyVDHa6uv8Fh0
3933,10,Reptilia,The Strokes,Room On Fire,91,162:46,spotify:track:57Xjny5yNzAcsxnusKmAfA



Wrapped 2018 – Top 10 Songs:


,rank,track_name,artist_name,album_name,total_play_count,listening_length,spotify_track_uri
6081,1,Star Treatment,Arctic Monkeys,Tranquility Base Hotel & Casino,180,786:51,spotify:track:0FgNSsaSZTvbLXUumSO8LQ
1604,2,Derka Blues,The Growlers,Hung At Heart,242,609:10,spotify:track:7xa1QzdX3wq5cnPk1UUwkn
3659,3,Leave It In My Dreams,The Voidz,Virtue,175,579:02,spotify:track:31u6rUeIEXGrYVoh10U7eu
3761,4,Like Real People Do,Hozier,Hozier,161,519:29,spotify:track:7K6LFPfjdnN6QqvGzhvpRO
1560,5,Days,The Drums,Portamento,120,475:23,spotify:track:6113aOfHIC0vbZVDZ6PpRV
629,6,Batphone,Arctic Monkeys,Tranquility Base Hotel & Casino,105,459:51,spotify:track:7seSDB6TiLZarbicyDIjiQ
6774,7,The World's First Ever Monster Truck Front Flip,Arctic Monkeys,Tranquility Base Hotel & Casino,136,352:21,spotify:track:1JdArJ9NKCF9TQASGQgszg
7098,8,Two Princes,Spin Doctors,Pocket Full Of Kryptonite,115,349:07,spotify:track:4ePP9So5xRzspjLFVVbj90
7594,9,Wink,The Voidz,Virtue,112,283:00,spotify:track:6VFaZ3zgXaRbc79GRqhbBz
3677,10,Lenny Sinpablo Juliano 36th,The Growlers,Are You In Or Out?,116,147:33,spotify:track:37FHss8Y0tCc3jCQhKWd9o



Wrapped 2019 – Top 10 Songs:


...

,rank,track_name,artist_name,album_name,total_play_count,listening_length,spotify_track_uri
8668,1,The Waiting,Angel Olsen,Half Way Home,226,651:05,spotify:track:1XGwjdHXHNu3842f75eg3T
2303,2,Drunk and with Dreams,Angel Olsen,Strange Cacti,117,379:25,spotify:track:2yDD2H5d0ZunPscZLp7MU3
7626,3,Small Talk,Julia Jacklin,Don't Let The Kids Win,130,366:53,spotify:track:750cDrvopZOV30PjHkWBTn
4896,4,Like I Used To,Sharon Van Etten,Like I Used To,120,353:41,spotify:track:1kZpYFQHUKv4xHELaaUSqP
1264,5,Caderas Blancas,Mon Laferte,Norma,116,319:05,spotify:track:4BMk6XhlXJW7ZgIS8vSBZk
6622,6,Pressure To Party,Julia Jacklin,Crushing,150,307:56,spotify:track:5Wyy2JHaM8cKEN6YDC6C8O
3252,7,Good For It,French Cassettes,Good For It,144,269:41,spotify:track:7GbxQwgFyfKuw6eaqseKmC
3938,8,I Like,Arlo Parks,Super Sad Generation,127,259:24,spotify:track:5BgfodYBmppmlfR8kEuBFT
7862,9,Spring,Angel Olsen,All Mirrors,135,245:04,spotify:track:3HjyXmT0PyqVFGHR97VnKu
2449,10,Embody,Frankie Cosmos,Next Thing,116,142:23,spotify:track:4jduNw2I6SPmPEiYTb7Jod



Wrapped 2023 – Top 10 Songs:


,rank,track_name,artist_name,album_name,total_play_count,listening_length,spotify_track_uri
4621,1,Hummingbird (Metro Boomin & James Blake),Metro Boomin,METRO BOOMIN PRESENTS SPIDER-MAN: ACROSS THE S...,75,290:47,spotify:track:6HexNTb392JS071DoTGo0y
11977,2,YUKON (INTERLUDE),Joji,SMITHEREENS,117,258:46,spotify:track:5IPl8JpkbtSH1mdyq5ctSx
111,3,3 Boys,Omar Apollo,3 Boys,86,242:38,spotify:track:31Wlc9ZnraX3JxrvMg9e8H
1295,4,Body Paint,Arctic Monkeys,The Car,68,231:32,spotify:track:42GuKw49pPxNAkIhWGwgFs
12266,5,dirty dancer,Orion Sun,dirty dancer,108,218:31,spotify:track:1ciQU7ZQGHq129m3njp9en
6525,6,McKenzie,Houndmouth,Good For You,65,181:00,spotify:track:6FLkXWDTvUc36qYYRhm4jg
3769,7,Ghost On,Angel Olsen,Big Time,68,171:01,spotify:track:512vqW0xDLcWjWD2tc46xd
8701,8,Run Right Back,The Black Keys,El Camino,76,164:57,spotify:track:5HgAZuHFAU5qLLMYuIQkgq
2385,9,Dearest,The Black Keys,Rave On Buddy Holly,70,158:49,spotify:track:32lewphnrnKx7lgsanoakd
5216,10,Intern,Angel Olsen,MY WOMAN,87,149:15,spotify:track:1IabnyjcfHmWwywryoQg2Q



Wrapped 2024 – Top 10 Songs:


,rank,track_name,artist_name,album_name,total_play_count,listening_length,spotify_track_uri
3984,1,Paddle to the Stars,The Dip,Sticking With It,125,317:09,spotify:track:0N3Gc2K38RkddGTWbFy2lD
2812,2,Jackie Big Tits,The Kooks,Inside In / Inside Out,95,221:13,spotify:track:3gd1MYbF3ZWePKIPVSh73X
4616,3,Shark Smile - Edit,Big Thief,Shark Smile,67,186:54,spotify:track:6exdwZ3EOSCjb11bd6k6Np
3759,4,Normal Day,French Cassettes,Benzene,62,176:29,spotify:track:1QUl57cQXPZkPvO89T88f2
4959,5,Stick Season,Noah Kahan,Stick Season,68,165:09,spotify:track:0mflMxspEfB0VbI1kyLiAv
1204,6,Davy Crochet,The Backseat Lovers,When We Were Friends,61,160:32,spotify:track:1w9B61OdLdnjzZIUYmy0bd
3023,7,Last Nite,The Strokes,Is This It,78,131:22,spotify:track:3SUusuA9jH1v6PVwtYMbdv
6010,8,White Noise,French Cassettes,Benzene,67,124:22,spotify:track:6gQVVUrx94Rx1jJHS5NCNJ
6208,9,You've Got To Hide Your Love Away - Remastered...,The Beatles,Help!,62,117:39,spotify:track:4F1AgKpuFRMLEgtPETVwZk
6192,10,You're Pretty Good Looking (For a Girl),The White Stripes,De Stijl,69,94:36,spotify:track:5OjQHOLMSzm9gkcIhtsgMO


In [6]:
# Title: Display Top 10 Artists per Year

# Description:
# Loads Spotify streaming data and converts timestamps, 
# loops through each year to identify the top 10 most-listened artists based on total playtime, 
# calculates play counts and listening duration in minutes and seconds, 
# ranks artists by total listening time, 
# formats and displays a clean yearly table with rank, artist name, listening length, and total plays.

In [7]:
# Load streaming data
streaming_data = pd.read_csv('wrapped_data/streaming_data.csv')
streaming_data['ts'] = pd.to_datetime(streaming_data['ts'])

# Top 10 Artists per Year
for year in sorted(streaming_data['year'].unique()):
    print(f"\nWrapped {year} – Top 10 Artists:")

    year_df = streaming_data.query("year == @year")

    if year_df.empty:
        print("No data for this year.")
        continue

    top_10_artists = (
        year_df
        .groupby('master_metadata_album_artist_name', as_index=False)
        .agg(
            total_ms_played=('ms_played', 'sum'),
            total_play_count=('ts', 'count')
        )
        .sort_values(by='total_ms_played', ascending=False)  # Rank by listening time
        .head(10)  # Top 10 only
        .assign(
            rank=lambda df: range(1, 11),
            total_listening_length=lambda df: df['total_ms_played'].apply(
                lambda ms: f"{int(ms // 60000)}:{int((ms % 60000) // 1000):02d}"
            )
        )
        .rename(columns={'master_metadata_album_artist_name': 'artist_name'})
        [['rank', 'artist_name', 'total_listening_length', 'total_play_count']]
    )

    display(top_10_artists)


Wrapped 2017 – Top 10 Artists:


,rank,artist_name,total_listening_length,total_play_count
1772,1,The Strokes,6720:06,3418
1685,2,The Growlers,6181:13,3197
277,3,Cage The Elephant,2688:10,1294
1597,4,Talking Heads,2343:21,702
1705,5,The Killers,2223:14,1086
97,6,Arctic Monkeys,1990:55,1088
841,7,Juanes,1693:50,935
1631,8,The Black Keys,1686:12,821
1017,9,Mac DeMarco,1248:18,648
933,10,Lana Del Rey,1195:08,422



Wrapped 2018 – Top 10 Artists:


,rank,artist_name,total_listening_length,total_play_count
2062,1,The Growlers,6240:18,3041
108,2,Arctic Monkeys,5900:29,2410
2161,3,The Strokes,3932:54,1719
2182,4,The Voidz,3074:14,1305
1165,5,Lana Del Rey,2834:57,921
1785,6,Sam Cooke,2599:21,1064
2150,7,The Smiths,2090:05,713
1094,8,Kanye West,1500:06,1017
2006,9,The Beatles,1444:48,793
919,10,Jack Johnson,1359:20,792



Wrapped 2019 – Top 10 Artists:


...

,rank,artist_name,total_listening_length,total_play_count
184,1,Angel Olsen,4165:03,1750
3658,2,The Black Keys,2127:13,1306
225,3,Arctic Monkeys,2112:51,1209
3757,4,The Growlers,1867:29,1089
3909,5,The Strokes,1551:27,953
1574,6,Houndmouth,1489:23,676
1582,7,Hozier,1405:25,542
1722,8,Japanese Breakfast,1300:11,641
691,9,Chicano Batman,1154:06,354
2440,10,Metro Boomin,1118:58,592



Wrapped 2024 – Top 10 Artists:


,rank,artist_name,total_listening_length,total_play_count
1279,1,Noah Kahan,1511:44,572
1791,2,The Growlers,1494:01,817
1886,3,The Strokes,1375:20,727
1770,4,The Dip,1262:42,534
628,5,French Cassettes,1225:49,662
826,6,Japanese Breakfast,999:33,393
763,7,Houndmouth,892:55,341
1518,8,Sammy Rae & The Friends,772:25,287
2018,9,Wallows,674:42,382
767,10,Hozier,655:13,249



Wrapped 2025 – Top 10 Artists:


,rank,artist_name,total_listening_length,total_play_count
313,1,Leon Bridges,184:23,84
389,2,Noah Kahan,117:18,59
610,3,Voxtrot,113:46,33
117,4,Charlie Burg,109:19,71
295,5,Khruangbin,101:03,34
468,6,Sam Cooke,87:27,51
29,7,Angel Olsen,78:58,36
316,8,Leslie Odom Jr.,72:27,71
414,9,Paul Simon,70:26,38
156,10,Declan McKenna,62:55,19


In [8]:
# Title: Identify Top 5 Genres per Year from Spotify Wrapped

# Description:
# Loads a cleaned artist-to-genre mapping and Spotify streaming history, 
# defines custom Wrapped cutoff dates for each year from 2017 to 2024, 
# filters listening data by year, 
# selects the top 200 most-played songs annually, 
# maps artists to genres using the cached genre dictionary, 
# counts genre frequency among top artists, 
# and prints the top 5 genres for each year based on artist appearances.

In [9]:

# Load the clean artist-genre map
CLEAN_CACHE_PATH = os.path.join('wrapped_data', 'top_genres_clean.json')

with open(CLEAN_CACHE_PATH, 'r') as f:
    artist_genre_map = json.load(f)

# Step 2: Load the streaming data
STREAMING_DATA_PATH = os.path.join('wrapped_data', 'streaming_data.csv')

streaming_data = pd.read_csv(STREAMING_DATA_PATH)
streaming_data['ts'] = pd.to_datetime(streaming_data['ts'])

# Step 3: Wrapped End Dates
wrapped_end_dates = {
    2017: "2017-10-31T23:59:59Z",
    2018: "2018-10-31T23:59:59Z",
    2019: "2019-10-31T23:59:59Z",
    2020: "2020-11-15T23:59:59Z",
    2021: "2021-11-15T23:59:59Z",
    2022: "2022-11-15T23:59:59Z",
    2023: "2023-11-15T23:59:59Z",
    2024: "2024-11-15T23:59:59Z"
}

# Calculate top genres per year
top_genres_by_year = {}

for year, end_str in wrapped_end_dates.items():
    print(f"\n=========================")
    print(f"Wrapped {year} – Top Genres")
    print("=========================")

    start = pd.Timestamp(f"{year}-01-01T00:00:00Z")
    end = pd.Timestamp(end_str)

    year_data = streaming_data[(streaming_data['ts'] >= start) & (streaming_data['ts'] <= end)]

    if year_data.empty:
        print(f"No data for {year}.")
        top_genres_by_year[year] = []
        continue

    # Get Top 200 Songs
    top_songs_candidates = (
        year_data
        .groupby(['master_metadata_track_name', 'master_metadata_album_artist_name'])
        .agg({'ts': 'count', 'ms_played': 'sum'})
        .reset_index()
        .rename(columns={'ts': 'play_count'})
        .sort_values(by='play_count', ascending=False)
        .head(200)
    )

    # Filter: Only want songs where artist has an identified genres
    valid_artists = []
    genres = []

    for artist in top_songs_candidates['master_metadata_album_artist_name'].dropna().unique():
        artist_genres = artist_genre_map.get(artist)
        
        if artist_genres:
            valid_artists.append(artist)
            genres += artist_genres
        
        if len(valid_artists) >= 100:
            break

    if not genres:
        print(f"No valid genres found for {year}.")
        top_genres_by_year[year] = []
        continue

    genre_counts = pd.Series(genres).value_counts().head(5)
    top_genres_by_year[year] = genre_counts

    # Print ranked list
    ranked_genres = genre_counts.index.tolist()
    
    for idx, genre in enumerate(ranked_genres, start=1):
        print(f"{idx}. {genre}")


Wrapped 2017 – Top Genres
1. indie
2. surf rock
3. garage rock
4. psychedelic pop
5. indie rock

Wrapped 2018 – Top Genres
1. indie
2. garage rock
3. surf rock
4. classic rock
5. baroque pop

Wrapped 2019 – Top Genres
1. indie
2. new wave
3. surf rock
4. garage rock
5. bedroom pop

Wrapped 2020 – Top Genres
1. indie rock
2. indie
3. surf rock
4. latin alternative
5. psychedelic pop

Wrapped 2021 – Top Genres
1. indie
2. bedroom pop
3. indie rock
4. garage rock
5. indie folk

Wrapped 2022 – Top Genres
1. bedroom pop
2. indie
3. garage rock
4. latin alternative
5. indie rock

Wrapped 2023 – Top Genres
1. indie
2. bedroom pop
3. alternative r&b
4. baroque pop
5. chamber pop

Wrapped 2024 – Top Genres
1. musicals
2. indie
3. retro soul
4. garage rock
5. indie folk
